In [3]:
#IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


import pandas as pd

from transformers import BertTokenizer, BertModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import joblib as jl

In [4]:
# MOUNTING DRIVE AND CHECKING COLAB CONNECTION

from google.colab import drive
drive.mount('/content/drive')
%ls
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive/  sample_data/
Wed May 13 00:19:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       3MiB /  15360MiB |      0%      

In [ ]:
# BERT CLASSIFIER DEFINITION

class BertClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained('bert-base-uncased')

        self.dropout = nn.Dropout(0.1)

        self.fc = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(cls_embedding)

        x = self.fc(x)

        return x

In [ ]:
#DATASET

df = pd.read_csv("/content/drive/MyDrive/SevenPhishingEmails/scikit_cleaned.csv")

texts = (
    df['sender'].fillna('') + ' ' +
    df['receiver'].fillna('') + ' ' +
    df['date'].fillna('') + ' ' +
    df['subject'].fillna('') + ' ' +
    df['body'].fillna('')
)

labels = df['label']

In [ ]:
# 60-20-20 Split

X_train, X_temp, y_train, y_temp = train_test_split(
    texts,
    labels,
    test_size=0.4,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [ ]:
# TOKENIZER

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# DATASET CLASS

class EmailDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):

        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=256,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [ ]:
# DATALOADERS

train_dataset = EmailDataset(X_train, y_train, tokenizer)
val_dataset = EmailDataset(X_val, y_val, tokenizer)
test_dataset = EmailDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
# MODEL, OPTIMIZER, LOSS FUNCTION

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertClassifier().to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

criterion = nn.BCEWithLogitsLoss()

In [ ]:
# TRAINING LOOP

EPOCHS = 3

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask).squeeze(1)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

In [ ]:
# VALIDATION

model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask)

        probs = torch.sigmoid(outputs)

        preds = (probs >= 0.5).int().cpu().numpy()

        predictions.extend(preds.flatten())

        actuals.extend(batch['label'].numpy())

val_acc = accuracy_score(actuals, predictions)

print("Validation Accuracy:", val_acc)

In [ ]:
# FINAL TESTING AND SAVING 

def final_test()
    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask)

            probs = torch.sigmoid(outputs)

            preds = (probs >= 0.5).int().cpu().numpy()

            predictions.extend(preds.flatten())

            actuals.extend(batch['label'].numpy())

    test_acc = accuracy_score(actuals, predictions)

    print("Test Accuracy:", test_acc)

    torch.save(model.state_dict(), "/content/drive/MyDrive/bert_model.pth")
    tokenizer.save_pretrained("/content/drive/MyDrive/bert_tokenizer")

    metadata = {
    "validation_accuracy": val_acc,
    "test_accuracy": test_acc
    }

    torch.save(metadata, "/content/drive/MyDrive/bert_metadata.pth")

In [ ]:
#again commented so don't accidentally run

#final_test() 